# 09 · Teacher cache (stage S8)

The LoRA (S9) learns to answer with fewer visual tokens by copying the frozen model with all of them. This notebook
stores that teacher: for every `lora_train` item and every loop count R = 1..5, the greedy answer and, for each answer
token, the 32 most likely tokens with their log-probabilities plus the mass of all the others.

Setup as in S2: no thinking, `ANSWER:` prefill, 512 px. The vision tower runs once per item and its output is reused
for every R; each R is generated on its own. Settings: `configs/train_lora.yaml`.

**How to run:** Run All, no edits: one item, then 20 items per benchmark with their checks. `RUN_FULL` is off.
**Paste the Step-2 output first.** The checks need one free GPU pair; the full run may use GPUs 0-7 (S8 only).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# GPU choice must precede CUDA init, which importing adaloop.api already does: the one import outside adaloop.api
from utils.gpus import select_for_notebook

CONFIG = "configs/train_lora.yaml"
GPU_IDS, N_COPIES = select_for_notebook(CONFIG)

In [ ]:
import math
import os
import socket

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook("configs/train_lora.yaml")
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
import torch

IS_INARA = "inara" in socket.gethostname()
print("host", socket.gethostname(), "| on Inara:", IS_INARA)
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
for i in range(torch.cuda.device_count()):
    total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i} | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")

In [ ]:
import pandas as pd

In [ ]:
# Settings
RUN_FULL = False  # set True only after the checks below passed; or use the terminal command in step 6
N_CHECK = cfg["sample_items"]  # steps 3-5: lora_train items per benchmark
R_LIST = cfg["teacher"]["R_list"]
TOP_N = cfg["teacher"]["top_n"]
TOL = cfg["teacher"]["check_prob_tol"]
CHECK_ROOT = ROOT / cfg["out_dir"] / "nb09"  # the checks write their own cache, apart from the full run's
CHECK_CFG = {**cfg, "teacher": {**cfg["teacher"], "out_dir": str(CHECK_ROOT)}}
print("full cache:", api.teacher_dir(cfg, ROOT))
cfg["teacher"]

### Step-1:- Load the model

In [ ]:
print("[STAGE] load model")
vlms = api.load_models(cfg, N_COPIES)
vlm = vlms[0]
tok = vlm.processor.tokenizer

### Step-2:- One item at R = 1..5 (paste this output)

The first GQA `lora_train` item: the teacher answer per R and, per answer token, the 5 most likely tokens with their
probabilities and the mass outside the top 32.

In [ ]:
one = api.teacher_items(cfg, ROOT, limit=1)[:1]
one[0].question, one[0].answers

In [ ]:
print("[STAGE] one item")
api.build_teacher_cache(vlms[:1], one, CHECK_CFG, ROOT)
one_rows, one_tensors = api.teacher_records(CHECK_CFG, ROOT, one)
for row, record in zip(one_rows, one_tensors):
    print(f"\nR={row['R']} | raw {row['raw_output']!r} | pred {row['pred']!r} | gold {row['answers'][:3]} | correct {row['correct']}")
    for step, token_id in enumerate(record["answer_ids"].tolist()):
        probs = record["top_logp"][step, :5].exp().tolist()
        top = ", ".join(f"{tok.decode([i], skip_special_tokens=False)!r} {p:.3f}"
                        for i, p in zip(record["top_ids"][step, :5].tolist(), probs))
        print(f"  token {step}: {tok.decode([token_id], skip_special_tokens=False)!r} | top5: {top} | tail {record['tail_logp'][step].exp():.2e}")

In [ ]:
assert [row["R"] for row in one_rows] == R_LIST, [row["R"] for row in one_rows]
for row, record in zip(one_rows, one_tensors):
    n = row["n_answer_tokens"]
    assert row["parse_ok"] and not row["hit_limit"], f"R={row['R']}: {row['raw_output']!r}"
    assert record["top_ids"].shape == record["top_logp"].shape == (n, TOP_N) and record["tail_logp"].shape == (n,)
    assert torch.equal(record["top_ids"][:, 0], record["answer_ids"]), f"R={row['R']}: the greedy token is not the top token"
print(f"✓ one item: {len(R_LIST)} records, parsed, shapes (n, {TOP_N}), greedy token = top token")

### Step-3:- 20 items per benchmark, then cache = live

The cache came from batched decoding with the KV cache. Here every record is recomputed with one fresh teacher-forced
forward on the prompt plus the cached answer ids (the way the student will be run). Tolerance: no top-32 token
probability differs by more than `check_prob_tol` (BF16 arithmetic differs between the two paths).

In [ ]:
sample = api.teacher_items(cfg, ROOT, limit=N_CHECK)
len(sample), pd.Series([it.benchmark for it in sample]).value_counts().to_dict()

In [ ]:
print("[STAGE] sample cache")
sample_run = api.build_teacher_cache(vlms, sample, CHECK_CFG, ROOT)
sample_rows, sample_tensors = api.teacher_records(CHECK_CFG, ROOT, sample)
print("[STAGE] cache = live")
live = pd.DataFrame(api.check_teacher_live(vlm, sample, CHECK_CFG, ROOT))
live.groupby("R")[["max_prob_diff", "max_logp_diff_top5", "argmax_match"]].agg(["mean", "max", "min"])

In [ ]:
assert len(sample_rows) == len(live) == len(sample) * len(R_LIST), (len(sample_rows), len(live))
worst = live.sort_values("max_prob_diff", ascending=False).head(3)
assert live.max_prob_diff.max() <= TOL, f"cache differs from live by more than {TOL}:\n{worst}"
print(f"✓ cache = live: largest top-token probability gap {live.max_prob_diff.max():.4f} (tolerance {TOL}), "
      f"largest top-5 log-prob gap {live.max_logp_diff_top5.max():.3f}, argmax = cached answer {100 * live.argmax_match.mean():.2f}%")

#### Probabilities sum to one

The top 32 plus the tail must be the whole distribution.

In [ ]:
totals = torch.cat([r["top_logp"].exp().sum(dim=-1) + r["tail_logp"].exp() for r in sample_tensors])
assert (totals - 1).abs().max() < 1e-4, (totals.min().item(), totals.max().item())
print(f"✓ top-{TOP_N} + tail = 1 on {len(totals)} answer tokens (range {totals.min():.6f} .. {totals.max():.6f})")

### Step-4:- Resume

Running a finished shard again must add nothing.

In [ ]:
again = api.build_teacher_cache(vlms, sample, CHECK_CFG, ROOT)
rows_after, _ = api.teacher_records(CHECK_CFG, ROOT, sample)
assert again["new_records"] == 0 and len(rows_after) == len(sample_rows), (again, len(rows_after))
print(f"✓ resume: no new records, still {len(rows_after)}")

### Step-5:- Teacher quality, speed and size

Accuracy per R, how often two loop counts give the same answer, throughput (this run only, if it computed anything) and
the storage of the full cache.

In [ ]:
per_R, pairs = api.teacher_summary(sample_rows)
pd.DataFrame(per_R)

In [ ]:
pd.DataFrame(pairs).pivot_table(index=["benchmark", "R_a"], columns="R_b", values="agreement").round(0)

In [ ]:
folder = sample_run["folder"]
size = sum(p.stat().st_size for p in folder.rglob("*") if p.is_file())
n_records = len(api.read_index(folder))
total = api.teacher_total_records(cfg, ROOT)
print(f"{size / n_records:.0f} bytes per record | full cache {total} records ≈ {size / n_records * total / 1e6:.0f} MB")
if sample_run["new_records"]:
    rate = sample_run["new_records"] / sample_run["seconds"]
    print(f"{rate:.2f} records/s on {N_COPIES} copy | full run ≈ {total / rate / 3600:.1f} h per copy, "
          f"{total / rate / 3600 / len(cfg['gpu_groups']):.1f} h on {len(cfg['gpu_groups'])} pairs")
else:
    print("throughput: nothing was computed in this run (resumed); delete nothing, read it from the first run")

### Step-6:- Full run

All `lora_train` items. Recommended from a terminal, one process per GPU pair (free this kernel's model first):

```
python scripts/mvp.py teacher --shard 0/4
python scripts/mvp.py teacher --shard 1/4
python scripts/mvp.py teacher --shard 2/4
python scripts/mvp.py teacher --shard 3/4
```

GPUs 0-7 are allowed for this run only. With fewer free pairs use fewer shards (`--shard i/2`); every run skips what is
already cached, whatever the shard count. `RUN_FULL = True` runs it in this kernel instead.

In [ ]:
if RUN_FULL:
    print("[STAGE] full teacher cache")
    items = api.teacher_items(cfg, ROOT)
    print(api.build_teacher_cache(vlms, items, cfg, ROOT))
else:
    print("skipped: RUN_FULL is False")

#### Count (also after a terminal run)

In [ ]:
full_rows = api.read_index(api.teacher_dir(cfg, ROOT))
total = api.teacher_total_records(cfg, ROOT)
print(f"{len(full_rows)} of {total} records cached in {api.teacher_dir(cfg, ROOT)}")

In [ ]:
if len(full_rows) == total:
    per_R, pairs = api.teacher_summary(full_rows)
    display(pd.DataFrame(per_R))
    assert len({(r['benchmark'], r['item_id'], r['R']) for r in full_rows}) == total
    print(f"✓ teacher cache complete: {total} records")
else:
    print("full cache not complete yet")